# PyBDR demo

[PyBDR](https://github.com/ASAG-ISCAS/PyBDR) computes reachable sets of nonlinear systems. With the
set-boundary based method only the **boundary** of the initial set is propagated, which avoids the
wrapping effect that makes the reachable sets of large initial sets explode.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ASAG-ISCAS/PyBDR/blob/master/examples/colab_demo.ipynb)

In [ ]:
# install PyBDR with the optional plotly dependency (skipped when it is already installed)
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pybdr") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pybdr[vis] @ git+https://github.com/ASAG-ISCAS/PyBDR"])

In [ ]:
import time

import numpy as np

import pybdr
from pybdr.algorithm import ASB2008CDC
from pybdr.geometry import Geometry, Interval, Polytope, Zonotope
from pybdr.geometry.operation import boundary, cvt2
from pybdr.model import brusselator, vanderpol
from pybdr.util.visualization import plot, plot_cmp, plot_tube

print("PyBDR", pybdr.__version__)

## 1. Van der Pol oscillator

$$\dot x = y, \qquad \dot y = (1 - x^2)\,y - x$$

The reachable sets are computed twice: from the whole initial box, and from the cells covering its
boundary (in parallel).

In [ ]:
options = ASB2008CDC.Options()
options.t_end = 3.0
options.step = 0.02
options.tensor_order = 3
options.taylor_terms = 4
options.u = Zonotope.zero(1, 1)
options.u_trans = np.zeros(1)

Zonotope.REDUCE_METHOD = Zonotope.REDUCE_METHOD.GIRARD
Zonotope.ORDER = 50

x0 = Interval([1.23, 2.34], [1.57, 2.46])
cells = boundary(x0, 0.1, Geometry.TYPE.ZONOTOPE)
print(len(cells), "boundary cells")

start = time.perf_counter()
ri_whole, rp_whole = ASB2008CDC.reach(vanderpol, [2, 1], options, cvt2(x0, Geometry.TYPE.ZONOTOPE))
print(f"whole initial set: {time.perf_counter() - start:.1f}s")

start = time.perf_counter()
ri_bound, rp_bound = ASB2008CDC.reach_parallel(vanderpol, [2, 1], options, cells)
print(f"boundary cells:    {time.perf_counter() - start:.1f}s")

In [ ]:
# orange: whole initial set, blue: boundary analysis
plot_cmp([ri_whole, ri_bound], [0, 1], cs=["#FF5722", "#303F9F"], width=600, height=600);

### Over time

`plot_tube` extrudes the reachable sets along the time axis. The figure is interactive: drag to rotate,
scroll to zoom.

In [ ]:
plot_tube(ri_bound[1:], [0, 1], step=options.step, width=700, height=600);

## 2. Polytope initial set

The boundary of polytopes and zonotopes is covered with small boxes by interval analysis (codac).

In [ ]:
a = np.array([[0.84680084, 0.53191008], [0.17672503, 0.98426026], [-0.68443591, 0.72907303],
              [-0.99997959, -0.00638965], [-0.92101281, -0.3895323], [-0.39691115, -0.91785704],
              [0.60973787, -0.79260314], [0.93952577, -0.34247821]])
b = np.array([0.40646441, 0.43238234, 0.45970287, 0.41445799, 0.40973516, 0.44057138, 0.38044951, 0.45097237])
x0 = Polytope(a, b)

boxes = boundary(x0, 0.08, Geometry.TYPE.INTERVAL)
print(len(boxes), "boundary boxes")
plot([x0, *boxes], [0, 1], width=500, height=500);

In [ ]:
options = ASB2008CDC.Options()
options.t_end = 4.0
options.step = 0.05
options.tensor_order = 2
options.taylor_terms = 4
options.u = Zonotope.zero(1, 1)
options.u_trans = np.zeros(1)

start = time.perf_counter()
cells = [cvt2(box, Geometry.TYPE.ZONOTOPE) for box in boxes]
ri, rp = ASB2008CDC.reach_parallel(brusselator, [2, 1], options, cells)
print(f"{time.perf_counter() - start:.1f}s")
plot(ri, [0, 1], width=600, height=600);

## 3. Your own system

Dynamics are written with sympy expressions of the state `x` and input `u`. `reach` runs in the
notebook process, so any function defined here can be used (`reach_parallel` needs picklable
functions, see the README).

In [ ]:
from sympy import Matrix


def pendulum(x, u):
    return Matrix([x[1], -0.2 * x[1] - 4.0 * x[0] + u[0]])


options = ASB2008CDC.Options()
options.t_end = 3.0
options.step = 0.02
options.tensor_order = 2
options.taylor_terms = 4
options.u = Zonotope.zero(1, 1)
options.u_trans = np.zeros(1)

ri, rp = ASB2008CDC.reach(pendulum, [2, 1], options, cvt2(Interval([0.8, -0.1], [1.0, 0.1]), Geometry.TYPE.ZONOTOPE))
plot_tube(ri[1:], [0, 1], step=options.step, width=700, height=600);